# 02 — Taxa manual

**Pergunta de negócio.** Para um grupo sem histórico próprio (cotação nova), qual a mortalidade esperada em
relação à BR-EMSmt 2026, dada a atividade do estipulante e o capital das vidas?

**Modelo.** GLMM de Poisson estimado no R com **glmmTMB**. Em cada célula $c$ (estipulante × faixa etária ×
sexo × faixa de capital), com $E_c$ óbitos esperados pela tábua:

$$
N_c \sim \text{Poisson}(\mu_c), \qquad
\log \mu_c = \log E_c + \beta_0 + \beta_{cap}\,\log\frac{\bar C_c}{C_{ref}} + \beta_{s(c)} + u_{d(c)} + v_{g(c)},
$$

$$
u_d \sim N(0, \sigma^2_{div}), \qquad v_g \sim N(0, \sigma^2_{est}).
$$

- $\beta_s$: efeito **fixo** da seção CNAE; $u_d$: efeito **aleatório** da divisão dentro da seção. É a estrutura
  de Ohlsson (2008) para fatores com muitos níveis: cada divisão recebe credibilidade em torno da sua seção.
- $v_g$: efeito aleatório do estipulante. Separa a atividade do grupo e dá intervalos corretos, porque células do
  mesmo grupo não são independentes.
- $\bar C_c$: capital médio da célula, ponderado pelos óbitos esperados.

**Decisões tomadas antes de olhar os dados** (`config/manual.yaml`):
1. Atividade e capital entram na tarifa por decisão de negócio.
2. Faixa etária, sexo e porte são candidatos: entram só com evidência (razão de verossimilhança, correção de Holm).
   Idade e sexo já estão na tábua; o teste mede o desvio adicional.
3. Estima-se em contagem de óbitos; o valor entra na renovação, com pooling.
4. Só entra experiência maturada (até 12 meses antes da data de referência).

In [1]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from vgpricing.config import carregar_config
from vgpricing.tabua import obter_tabua
from vgpricing import manual as M, glmm as G

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
cfg, cm = carregar_config("base", RAIZ), carregar_config("manual", RAIZ)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*coluna lx.*")
    tabua = obter_tabua(cfg, RAIZ)
assert not tabua.placeholder and G.r_disponivel(), "Precisa da tábua oficial e do R com glmmTMB (veja o README)"
limpo = RAIZ / cfg["pasta_processados"] / "limpo"
dados = {t: pd.read_parquet(limpo / f"{t}.parquet") for t in ("estipulantes", "vidas", "sinistros")}
cel, info = M.base_modelagem(dados["vidas"], dados["sinistros"], dados["estipulantes"], tabua,
                             cfg["data_referencia"], cm, cfg["exposicao"]["dias_por_ano"])
pd.Series({k: v for k, v in info.items() if k != "ids_sem_celula"}).to_frame("base de modelagem")

,base de modelagem
corte_experiencia,2024-12-31
sinistros_no_periodo,1725
sinistros_sem_celula,30
celulas_sexo_ND_removidas,1206
celulas,6463
estipulantes,250
obitos,1686
esperados,"1,627.3897"
secoes_agrupadas_em_OUTRAS,[]


Sinistros "sem célula" são de vidas sem exposição limpa na data do óbito; ficam listados em
`info["ids_sem_celula"]` para investigação.

## 1. Primeira leitura: A/E sem ajuste

In [2]:
uni = pd.concat([cel.groupby(f, observed=True)[["E", "N"]].sum().assign(fator=f).rename_axis("nivel").reset_index()
                 for f in ["cnae_secao", "faixa_capital", "faixa_idade", "sexo", "faixa_tamanho"]])
uni["A/E"] = uni["N"] / uni["E"]
uni.set_index(["fator", "nivel"])

E     N    A/E
fator         nivel                            
cnae_secao    C           200.3533   218 1.0881
              F           150.9667   186 1.2321
              G            68.1254    77 1.1303
              H           206.7769   300 1.4508
              K           133.3037   131 0.9827
              M            88.2555    68 0.7705
              N           277.0932   293 1.0574
              O           319.6548   254 0.7946
              P           116.8503    96 0.8216
              Q            66.0099    63 0.9544
faixa_capital 150-400k    181.3493   166 0.9154
              30-60k      493.2468   505 1.0238
              60-150k     453.8571   418 0.9210
              acima400k    27.0336    28 1.0357
              ate30k      471.9029   569 1.2058
faixa_idade   30-39       286.2026   282 0.9853
              40-49       496.2902   499 1.0055
              50-59       475.4007   497 1.0454
              60+         243.7276   273 1.1201
              ate29       125.7687   135 1.0734
sexo          F           526.3359   498 0.9462
              M         1,101.0538  1188 1.0790
faixa_tamanho 100-499     152.3829   175 1.1484
              2000+       814.2508   788 0.9678
              500-1999    635.2679   705 1.1098
              ate99        25.4881    18 0.7062

## 2. Ajuste e escolha dos candidatos

O modelo completo (obrigatórios + candidatos) é ajustado; cada candidato é testado retirando-o do modelo, pela
estatística $LR = 2(\ell_{completo} - \ell_{reduzido}) \sim \chi^2_{gl}$, e os p-valores são corrigidos por Holm.
Os efeitos aleatórios também são testados; como a variância fica na fronteira ($\sigma^2 = 0$) sob a hipótese
nula, o p-valor usa a mistura $\tfrac12\chi^2_0 + \tfrac12\chi^2_1$. Eles ficam no modelo de qualquer forma.

In [3]:
final, completo, testes = M.ajustar_taxa_manual(cel, cm)
print("Convergiu:", final.convergiu)
print("Fórmula final:", final.formula)
testes

Convergiu: True
Fórmula final: N ~ log_capital_c + cnae_secao + (1 | cnae_secao:cnae_divisao) + (1 | id_estipulante) + offset(log(E))


,termo,tipo,lr,gl,p_valor,p_holm,decisao
0,faixa_idade,fixo,5.5732,4,0.2334,0.4667,não entra
1,sexo,fixo,0.0054,1,0.9412,0.9412,não entra
2,faixa_tamanho,fixo,7.8721,3,0.0487,0.1462,não entra
3,(1|cnae_secao:cnae_divisao),aleatorio,-0.0000,1,0.5000,NaN,estrutura (mantido)
4,(1|id_estipulante),aleatorio,10.1349,1,0.0007,NaN,estrutura (mantido)


## 3. Onde está a sobredispersão?

In [4]:
M.comparar_dispersao(cel, cm, final)

,modelo,dispersão,loglik,parametros,AIC,dif_AIC_vs_melhor
0,GLM Poisson,nenhuma,"-2,615.4244",11,"5,252.8488",11.6349
1,Binomial Negativa (NB2),"por célula, independente","-2,615.0773",12,"5,254.1546",12.9408
2,GLMM (efeitos de divisão e estipulante),por grupo e por atividade,"-2,607.6069",13,"5,241.2138",0.0000


As três versões têm os mesmos efeitos fixos e são ajustadas no mesmo pacote (glmmTMB), então as
log-verossimilhanças são diretamente comparáveis. A Binomial Negativa trata a sobredispersão como ruído
independente por célula e quase não melhora a Poisson. O GLMM a coloca onde ela está, no grupo, e tem AIC menor.

## 4. Resultados

In [5]:
fx = final.fixos.assign(fator=lambda d: np.exp(d["estimativa"]),
                        ic95_inf=lambda d: np.exp(d["estimativa"] - 1.96 * d["ep"]),
                        ic95_sup=lambda d: np.exp(d["estimativa"] + 1.96 * d["ep"]))
print("Seção base (maior exposição):", final.especificacao["niveis_base"]["cnae_secao"])
fx

Seção base (maior exposição): O


,nome,estimativa,ep,fator,ic95_inf,ic95_sup
0,(Intercept),-0.3063,0.0844,0.7362,0.6239,0.8687
1,log_capital_c,-0.0863,0.0326,0.9173,0.8605,0.9778
2,cnae_secaoC,0.3468,0.1202,1.4146,1.1178,1.7902
3,cnae_secaoF,0.5200,0.1295,1.6820,1.3048,2.1681
4,cnae_secaoG,0.4019,0.1555,1.4947,1.1021,2.0272
5,cnae_secaoH,0.6393,0.1192,1.8951,1.5002,2.3939
6,cnae_secaoK,0.2570,0.1416,1.2930,0.9797,1.7065
7,cnae_secaoM,0.0837,0.1755,1.0873,0.7709,1.5336
8,cnae_secaoN,0.3582,0.1131,1.4307,1.1463,1.7857
9,cnae_secaoP,0.0908,0.1549,1.0951,0.8083,1.4836


In [6]:
tv = pd.DataFrame({"variância": final.variancias,
                   "fator marginal exp(σ²/2)": {k: np.exp(x / 2) for k, x in final.variancias.items()}})
if final.ic_variancias is not None:
    tv = tv.join(final.ic_variancias.set_index("termo"))
tv

,variância,fator marginal exp(σ²/2),var_inf,var_sup
cnae_secao:cnae_divisao,0.0000,1.0000,0.0000,Inf
id_estipulante,0.0485,1.0245,0.0232,0.1015


**Leitura:**
- **Capital:** dobrar o capital multiplica a mortalidade por $2^{\beta_{cap}}$. Coeficiente negativo: vidas de
  capital maior morrem menos.
- **Estipulante:** $\sigma^2_{est}$ mede quanto os grupos diferem além de atividade e capital; é o insumo da
  credibilidade da renovação. O intervalo é largo: com poucos óbitos por grupo, a variância entre grupos é mal
  determinada. Propagar essa incerteza para o preço é o papel da camada bayesiana.
- **Divisão:** variância perto de zero significa que, com este volume, as divisões não se distinguem das suas
  seções; cada divisão fica com o fator da seção até haver dados.
- **Grupo novo:** com efeito Normal na escala log, o fator esperado de um grupo desconhecido é
  $E[e^{v}] = e^{\sigma^2_{est}/2}$, não 1. A tabela de fatores já inclui essa correção.

## 5. A complexidade muda o preço?

Validação cruzada deixando **grupos inteiros** de fora (5 partes): cada modelo é ajustado sem um quinto dos
grupos e prevê o total de óbitos de cada grupo deixado de fora, como numa cotação de grupo novo. Métrica:
deviance de Poisson no total de cada grupo,

$$
D = 2\sum_g \left[N_g \log\frac{N_g}{\hat\mu_g} - (N_g - \hat\mu_g)\right],
$$

(menor é melhor), e a razão previstos/observados (viés).

In [7]:
fixos_final = final.especificacao["fixos"]
previsores = {
    "BR-EMS pura (A/E = 1)": lambda tr, te: te["E"].to_numpy(),
    "GLM Poisson (mesmos efeitos fixos, sem efeitos aleatórios)":
        lambda tr, te: G.prever(M.ajustar_comparacao(tr, cm, fixos_final, "poisson"), te),
    "GLMM (modelo final)": lambda tr, te: G.prever(M._ajustar(tr, cm, fixos_final), te),
}
cv = M.validacao_cruzada_grupos(cel, previsores, k=cm["validacao_cruzada"]["folds"], semente=cm["validacao_cruzada"]["semente"])
cv.sort_values("deviance_grupos_novos")

,modelo,deviance_grupos_novos,previstos/observados,dif_vs_melhor
1,"GLM Poisson (mesmos efeitos fixos, sem efeitos...",386.1924,1.0199,0.0000
2,GLMM (modelo final),387.2328,1.0246,1.0404
0,BR-EMS pura (A/E = 1),431.0929,0.9652,44.9006


Atividade e capital melhoram muito a previsão de grupos novos em relação à tábua pura. GLM e GLMM empatam no
ponto estimado; a diferença entre eles está nos intervalos (o GLM os subestima) e no encolhimento das divisões.
A validação definitiva, com o preço que seria cobrado, é o backtest fora do tempo da Etapa 3.

## 6. Tabela de fatores para o cotador

In [8]:
meta = {"gerado_em": pd.Timestamp.today().date().isoformat(), "tabua": tabua.nome, "tabua_sha256": tabua.sha256,
        "corte_experiencia": info["corte_experiencia"], "obitos": info["obitos"],
        "esperados_tabua": round(info["esperados"], 2), "software": "R glmmTMB",
        "candidatos_testados": cm["modelo"]["fixos_candidatos"],
        "candidatos_que_entraram": [f for f in fixos_final if f in cm["modelo"]["fixos_candidatos"]],
        "secoes_agrupadas_em_OUTRAS": info["secoes_agrupadas_em_OUTRAS"],
        "esperados_por_secao": cel.groupby("cnae_secao")["E"].sum().round(3).to_dict()}
tabela = M.tabela_fatores(final, cm, meta)
M.salvar_tabela(tabela, RAIZ / "artefatos/fatores_manual.yaml")
pd.Series({"A/E base (grupo novo, seção base, capital de referência)": tabela["ae_base"],
           "fator ao dobrar o capital": 2 ** tabela["capital"]["elasticidade"],
           "fator para seção sem experiência": tabela["secao_nao_vista"],
           "fator para divisão sem experiência": tabela["divisao_nao_vista"]}).to_frame("valor")

,valor
"A/E base (grupo novo, seção base, capital de referência)",0.7543
fator ao dobrar o capital,0.9419
fator para seção sem experiência,1.3683
fator para divisão sem experiência,1.0000


## 7. Exemplo: taxa pura de um grupo

Para cada vida $v$ do censo, com $m_v$ o produto dos fatores da tabela:

$$
\text{taxa pura anual} = 1000 \times \frac{\sum_v C_v\, q_{x_v}\, m_v}{\sum_v C_v} \quad (\text{por mil de capital}).
$$

In [9]:
ref = pd.Timestamp(cfg["data_referencia"])
ativos = dados["vidas"][dados["vidas"]["data_fim"].isna()]
porte = ativos["id_estipulante"].value_counts()
grupo = porte.index[len(porte) // 2]
e = dados["estipulantes"].set_index("id_estipulante").loc[grupo]
cot = M.taxa_pura_manual(ativos[ativos["id_estipulante"] == grupo], e["cnae_secao"], e["cnae_divisao"], tabela, tabua,
                         ref + pd.Timedelta(days=1))
pd.Series({"estipulante": grupo, "atividade": f"{e['cnae_secao']}:{e['cnae_divisao']}", "vidas": cot["vidas"],
           "fator de atividade": cot["fator_atividade"], "prêmio puro anual (R$)": cot["premio_puro_anual"],
           "taxa pura mensal (‰)": cot["taxa_pura_mensal_mil"], "avisos": "; ".join(cot["avisos"]) or "nenhum"}).to_frame("cotação")

,cotação
estipulante,EST0247
atividade,P:85
vidas,451
fator de atividade,1.0951
prêmio puro anual (R$),"89,542.3977"
taxa pura mensal (‰),0.1312
avisos,nenhum


## Limitações
- **Poder estatístico:** com poucos milhares de óbitos, efeitos pequenos (divisão, capital) ficam perto do ruído;
  o encolhimento cuida disso e, com mais volume, os fatores se afastam de 1 sem decisão nova.
- **Incerteza dos parâmetros:** as variâncias são tratadas como conhecidas na previsão (camada bayesiana).
- **Seções raras** são agrupadas em "OUTRAS" (limite no config).
- A verificação de que o estimador recupera parâmetros conhecidos está nos testes (`tests/test_glmm.py`) e no
  notebook `90_validacao_sintetica`.